# PrediRuta: carga de las tablas depuradas a Supabase


Este notebook carga primero las seis tablas relacionales depuradas en el proyecto principal de Supabase usando `host`, `port`, `database`, `user` y `password`.

Al final se incluye un bloque independiente para cargar `TABLA_COMPLETA.csv` en un segundo proyecto usando exclusivamente `host2`, `port2`, `database2`, `user2` y `password2`.

## 0. Configuración y conexión


En esta etapa se preparan las dependencias necesarias, se define la ubicación de los archivos CSV y se conserva la conexión principal con las variables `host`, `port`, `database`, `user` y `password`. La conexión del segundo proyecto para `TABLA_COMPLETA` se configura posteriormente en una celda independiente.

La contraseña se utiliza únicamente para establecer la conexión con la base de datos y no se muestra durante la ejecución.

In [ ]:
# Importar librerías necesarias
from pathlib import Path
from io import StringIO
import os
import time
import pandas as pd
import psycopg
from psycopg import sql
from dotenv import load_dotenv
from IPython.display import display

In [2]:
# Cargar las variables de conexión almacenadas en .env
load_dotenv()

# Configuración de rutas y datos de conexión
CARPETA_CSV = Path('data_procesada/tablas_finales')

SUPABASE_HOST = os.getenv('host') or os.getenv('SUPABASE_HOST')
SUPABASE_PORT = int(os.getenv('port') or os.getenv('SUPABASE_PORT', '5432'))
SUPABASE_USER = os.getenv('user') or os.getenv('SUPABASE_USER', 'postgres')
SUPABASE_DB = os.getenv('database') or os.getenv('SUPABASE_DB', 'postgres')
SUPABASE_PASS = os.getenv('password') or os.getenv('SUPABASE_PASS')

print('Configuración de conexión cargada correctamente.')

Configuración de conexión cargada correctamente.


## 1. Revisión de los archivos CSV


Antes de crear las tablas relacionales se verifica que los seis CSV estén disponibles y que sus columnas coincidan con el esquema depurado.

In [3]:
# Definición de archivos y columnas esperadas
ARCHIVOS = {
    'clima': 'CLIMA.csv',
    'accidente': 'ACCIDENTE.csv',
    'via': 'VIA.csv',
    'vehiculo': 'VEHICULO.csv',
    'actor_vial': 'ACTOR_VIAL.csv',
    'causa': 'CAUSA.csv',
}

COLUMNAS = {
    'clima': ['CLIMA_ID','FECHA_HORA_CLIMA','TEMPERATURA_2M','HUMEDAD_RELATIVA_2M','SENSACION_TERMICA','PRECIPITACION','LLUVIA','NUBOSIDAD','PRESION_SUPERFICIE','VELOCIDAD_VIENTO_10M','DIRECCION_VIENTO_10M'],
    'accidente': ['ACCIDENTE_ID','FECHA_HORA','LATITUD','LONGITUD','CLASE_ACCIDENTE','LOCALIDAD','OBJETIVO_GRAVE','CLIMA_ID'],
    'via': ['ACCIDENTE_ID','GEOMETRIA_PLANTA','GEOMETRIA_TERRENO','GEOMETRIA_SECCION','SENTIDO_VIA','N_CALZADAS','N_CARRILES','SUPERFICIE_RODADURA','ESTADO_VIA','CONDICION_VIA','ILUMINACION_ARTIFICIAL','SEMAFORO'],
    'vehiculo': ['ACCIDENTE_ID','VEHICULO_ID','CLASE','SERVICIO'],
    'actor_vial': ['ACCIDENTE_ID','ACTOR_ID','VEHICULO_ID','CONDICION','ESTADO','GENERO','EDAD'],
    'causa': ['CAUSA_ID','ACCIDENTE_ID','VEHICULO_ID','CODIGO_CAUSA','NOMBRE','TIPO'],
}

In [4]:
# Validar archivos CSV y sus columnas
for tabla, archivo in ARCHIVOS.items():
    ruta = CARPETA_CSV / archivo

    columnas = pd.read_csv(ruta, nrows=0, encoding='utf-8-sig').columns.tolist()

    if columnas != COLUMNAS[tabla]:
        raise ValueError(f'Las columnas de {archivo} no coinciden con las esperadas.')

print('Archivos CSV y columnas validados correctamente.')

Archivos CSV y columnas validados correctamente.


## 2. Prueba de conexión con Supabase


Primero se conserva y prueba la conexión principal. La carga consolidada usará posteriormente una conexión diferente y explícita para el segundo proyecto.

In [5]:
# Configurar y probar la conexión con Supabase
CONEXION = {
    'host': SUPABASE_HOST,
    'port': SUPABASE_PORT,
    'dbname': SUPABASE_DB,
    'user': SUPABASE_USER,
    'password': SUPABASE_PASS,
    'sslmode': 'require',
    'connect_timeout': 20,
}

with psycopg.connect(**CONEXION) as conexion:
    version = conexion.execute('SELECT version()').fetchone()[0]

print('Conexión correcta:', version.split(',')[0])

Conexión correcta: PostgreSQL 17.6 on x86_64-pc-linux-gnu


## 3. Creación de las tablas relacionales en el proyecto principal


Las seis tablas depuradas se crean en el proyecto principal, respetando el orden de sus relaciones.

Si una tabla ya existe, se mantiene. La carga consolidada del segundo proyecto aparece como una sección adicional al final del notebook.

In [6]:
# Crear las tablas y relaciones del modelo final en PostgreSQL

DDL = {
    'clima': '''CREATE TABLE IF NOT EXISTS clima (
        clima_id integer PRIMARY KEY,
        fecha_hora_clima timestamp NOT NULL,
        temperatura_2m double precision,
        humedad_relativa_2m double precision,
        sensacion_termica double precision,
        precipitacion double precision,
        lluvia double precision,
        nubosidad double precision,
        presion_superficie double precision,
        velocidad_viento_10m double precision,
        direccion_viento_10m double precision
    )''',
    'accidente': '''CREATE TABLE IF NOT EXISTS accidente (
        accidente_id integer PRIMARY KEY,
        fecha_hora timestamp NOT NULL,
        latitud double precision NOT NULL,
        longitud double precision NOT NULL,
        clase_accidente text,
        localidad text,
        objetivo_grave boolean NOT NULL,
        clima_id integer REFERENCES clima(clima_id)
    )''',
    'via': '''CREATE TABLE IF NOT EXISTS via (
        accidente_id integer PRIMARY KEY REFERENCES accidente(accidente_id),
        geometria_planta text,
        geometria_terreno text,
        geometria_seccion text,
        sentido_via text,
        n_calzadas numeric,
        n_carriles numeric,
        superficie_rodadura text,
        estado_via text,
        condicion_via text,
        iluminacion_artificial text,
        semaforo text
    )''',
    'vehiculo': '''CREATE TABLE IF NOT EXISTS vehiculo (
        accidente_id integer NOT NULL REFERENCES accidente(accidente_id),
        vehiculo_id integer PRIMARY KEY,
        clase text,
        servicio text
    )''',
    'actor_vial': '''CREATE TABLE IF NOT EXISTS actor_vial (
        accidente_id integer NOT NULL REFERENCES accidente(accidente_id),
        actor_id integer PRIMARY KEY,
        vehiculo_id integer REFERENCES vehiculo(vehiculo_id),
        condicion text,
        estado text,
        genero text,
        edad numeric
    )''',
    'causa': '''CREATE TABLE IF NOT EXISTS causa (
        causa_id integer PRIMARY KEY,
        accidente_id integer NOT NULL REFERENCES accidente(accidente_id),
        vehiculo_id integer REFERENCES vehiculo(vehiculo_id),
        codigo_causa text NOT NULL,
        nombre text,
        tipo text
    )''',
}

with psycopg.connect(**CONEXION) as conexion:
    for sentencia in DDL.values():
        conexion.execute(sentencia)

    # Ajustar tablas creadas en una ejecución anterior con tipos incompatibles con los CSV.
    conexion.execute('''ALTER TABLE via
        ALTER COLUMN n_calzadas TYPE numeric USING n_calzadas::numeric,
        ALTER COLUMN n_carriles TYPE numeric USING n_carriles::numeric''')
    conexion.execute('''ALTER TABLE actor_vial
        ALTER COLUMN edad TYPE numeric USING edad::numeric''')

print('Estructura de la base creada correctamente en Supabase.')


Estructura de la base creada correctamente en Supabase.


## 4. Revisión del estado actual de las tablas


Antes de iniciar la carga se revisa cuántos registros existen actualmente en cada tabla de Supabase. Si una tabla ya contiene información, se omite para evitar duplicar o sobrescribir datos.

Si alguna carga anterior quedó incompleta, primero se debe revisar su contenido antes de decidir si es necesario eliminarla y volver a cargarla.

In [7]:
# Revisar la cantidad de registros existentes en cada tabla
with psycopg.connect(**CONEXION) as conexion:
    estado_inicial = pd.DataFrame([
        {
            'tabla': tabla,
            'filas_en_supabase': conexion.execute(
                sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
            ).fetchone()[0]
        }
        for tabla in ARCHIVOS
    ])

display(estado_inicial)


,tabla,filas_en_supabase
0,clima,0
1,accidente,0
2,via,0
3,vehiculo,0
4,actor_vial,0
5,causa,0


## 5. Carga de los archivos CSV


Los archivos se cargan en lotes de `5.000` filas y respetando el orden de las relaciones. Cada lote se copia primero a una tabla temporal y luego se incorpora a la tabla definitiva sin repetir llaves existentes. Esta estrategia reduce el espacio transitorio requerido por PostgreSQL y permite reanudar una tabla parcialmente cargada. Durante la ejecución se muestra el avance y las tablas que ya coinciden con su CSV se omiten.


In [8]:
# Cargar los CSV por lotes en el orden definido por las relaciones
ORDEN_CARGA = ['clima', 'accidente', 'via', 'vehiculo', 'actor_vial', 'causa']
# Lotes pequeños para limitar el uso de memoria, WAL y Disk I/O en Supabase.
FILAS_POR_LOTE = 5_000
resultado_carga = []

def contar_filas_archivo(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(1 for _ in archivo) - 1, 0)

with psycopg.connect(**CONEXION) as conexion:
    for numero, tabla in enumerate(ORDEN_CARGA, 1):
        ruta = CARPETA_CSV / ARCHIVOS[tabla]
        filas_csv = contar_filas_archivo(ruta)
        existentes = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]

        if existentes == filas_csv:
            print(f'{numero}/{len(ORDEN_CARGA)} {tabla}: omitida ({existentes:,} filas completas)')
            resultado_carga.append([tabla, 'omitida', existentes, 0])
            continue

        columnas = [columna.lower() for columna in COLUMNAS[tabla]]
        temporal = f'_carga_{tabla}'
        conexion.execute(
            sql.SQL('CREATE TEMP TABLE {} (LIKE {} INCLUDING DEFAULTS) ON COMMIT DELETE ROWS').format(
                sql.Identifier(temporal),
                sql.Identifier(tabla),
            )
        )

        consulta_copy = sql.SQL(
            'COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE, ENCODING UTF8)'
        ).format(
            sql.Identifier(temporal),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
        )
        consulta_insertar = sql.SQL(
            'INSERT INTO {} ({}) SELECT {} FROM {} ON CONFLICT DO NOTHING'
        ).format(
            sql.Identifier(tabla),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.Identifier(temporal),
        )

        inicio = time.time()
        cargadas = existentes
        for lote, bloque in enumerate(pd.read_csv(
            ruta,
            dtype='string',
            encoding='utf-8-sig',
            chunksize=FILAS_POR_LOTE,
            low_memory=False,
        ), 1):
            buffer = StringIO()
            bloque.to_csv(buffer, index=False)
            buffer.seek(0)

            with conexion.cursor().copy(consulta_copy) as copia:
                copia.write(buffer.getvalue())
            resultado_insercion = conexion.execute(consulta_insertar)
            cargadas += resultado_insercion.rowcount
            conexion.commit()

            print(
                f'\r{numero}/{len(ORDEN_CARGA)} {tabla}: '
                f'{cargadas:,} / {filas_csv:,}',
                end='',
                flush=True,
            )

        tiempo = round(time.time() - inicio, 1)
        cargadas = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]
        print(f' en {tiempo} s')
        resultado_carga.append([tabla, 'completa' if cargadas == filas_csv else 'incompleta', cargadas, tiempo])

display(pd.DataFrame(
    resultado_carga,
    columns=['tabla', 'resultado', 'filas_en_supabase', 'segundos'],
))


1/6 clima: 325,314 / 325,314 en 37.0 s
2/6 accidente: 895,833 / 895,833 en 114.7 s
3/6 via: 488,662 / 488,662 en 61.7 s
4/6 vehiculo: 1,670,699 / 1,670,699 en 221.7 s
5/6 actor_vial: 1,928,580 / 1,928,580 en 287.7 s
6/6 causa: 1,284,682 / 1,284,682 en 187.1 s


,tabla,resultado,filas_en_supabase,segundos
0,clima,completa,325314,37.0
1,accidente,completa,895833,114.7
2,via,completa,488662,61.7
3,vehiculo,completa,1670699,221.7
4,actor_vial,completa,1928580,287.7
5,causa,completa,1284682,187.1


## 6. Validación de la carga


Finalmente se compara la cantidad de registros de cada CSV con los almacenados en Supabase. También se revisa que las tablas relacionadas con `ACCIDENTE` no tengan referencias huérfanas y que los `CLIMA_ID` utilizados existan en la tabla `CLIMA`.

La carga se considera completa cuando las cantidades coinciden entre los archivos y la base de datos, y no se encuentran relaciones inválidas entre las tablas.

In [9]:
# Validar cantidades de registros y relaciones entre tablas
def contar_filas_csv(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(bloque.count(b'\n') for bloque in iter(
            lambda: archivo.read(1024 * 1024), b''
        )) - 1, 0)

# Validar cantidades de registros y relaciones entre tablas
with psycopg.connect(**CONEXION) as conexion:

    # Comparar cantidad de filas entre los CSV y Supabase
    cantidades = []

    for tabla, archivo in ARCHIVOS.items():
        filas_csv = contar_filas_csv(CARPETA_CSV / archivo)
        filas_bd = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]

        cantidades.append([tabla, filas_csv, filas_bd, filas_csv == filas_bd])

    # Validar referencias hacia ACCIDENTE
    huerfanos = {}

    for tabla in ['via', 'vehiculo', 'actor_vial', 'causa']:
        huerfanos[tabla] = conexion.execute(
            sql.SQL('''
                SELECT COUNT(*)
                FROM {} t
                LEFT JOIN accidente a USING (accidente_id)
                WHERE a.accidente_id IS NULL
            ''').format(sql.Identifier(tabla))
        ).fetchone()[0]

    # Validar referencias de ACCIDENTE hacia CLIMA
    huerfanos['accidente_clima'] = conexion.execute('''
        SELECT COUNT(*)
        FROM accidente a
        LEFT JOIN clima c USING (clima_id)
        WHERE a.clima_id IS NOT NULL
          AND c.clima_id IS NULL
    ''').fetchone()[0]

display(pd.DataFrame(
    cantidades,
    columns=['tabla', 'filas_csv', 'filas_supabase', 'coincide']
))

display(pd.DataFrame(
    huerfanos.items(),
    columns=['relacion', 'referencias_huerfanas']
))


,tabla,filas_csv,filas_supabase,coincide
0,clima,325314,325314,True
1,accidente,895833,895833,True
2,via,488662,488662,True
3,vehiculo,1670699,1670699,True
4,actor_vial,1928580,1928580,True
5,causa,1284682,1284682,True


,relacion,referencias_huerfanas
0,via,0
1,vehiculo,0
2,actor_vial,0
3,causa,0
4,accidente_clima,0


## 7. Carga de `TABLA_COMPLETA` en el segundo proyecto

Se cargará a un nuevo proyecto la `TABLA_COMPLETA.csv`. La carga se realiza en lotes pequeños, puede reanudarse y se detiene preventivamente antes de acercarse al límite de almacenamiento.


In [3]:
# Validar el archivo consolidado sin modificar la configuración relacional anterior
ARCHIVO_TABLA_COMPLETA = CARPETA_CSV / 'TABLA_COMPLETA.csv'
COLUMNAS_TABLA_COMPLETA = ['ACCIDENTE_ID', 'FECHA_HORA', 'LATITUD', 'LONGITUD', 'CLASE_ACCIDENTE', 'LOCALIDAD', 'OBJETIVO_GRAVE', 'CLIMA_ID', 'FECHA_HORA_CLIMA', 'TEMPERATURA_2M', 'HUMEDAD_RELATIVA_2M', 'SENSACION_TERMICA', 'PRECIPITACION', 'LLUVIA', 'NUBOSIDAD', 'PRESION_SUPERFICIE', 'VELOCIDAD_VIENTO_10M', 'DIRECCION_VIENTO_10M', 'GEOMETRIA_PLANTA', 'GEOMETRIA_TERRENO', 'GEOMETRIA_SECCION', 'SENTIDO_VIA', 'N_CALZADAS', 'N_CARRILES', 'SUPERFICIE_RODADURA', 'ESTADO_VIA', 'CONDICION_VIA', 'ILUMINACION_ARTIFICIAL', 'SEMAFORO', 'CANTIDAD_VEHICULOS', 'CLASES_VEHICULO_DISTINTAS', 'SERVICIOS_VEHICULO_DISTINTOS', 'TIENE_MOTOCICLETA', 'TIENE_BICICLETA', 'TIENE_AUTOMOVIL', 'TIENE_BUS', 'TIENE_CAMION', 'CANTIDAD_ACTORES', 'EDAD_PROMEDIO', 'EDAD_MINIMA', 'EDAD_MAXIMA', 'CANTIDAD_CONDUCTORES', 'CANTIDAD_PASAJEROS', 'CANTIDAD_PEATONES', 'CANTIDAD_HERIDOS', 'CANTIDAD_MUERTOS', 'CANTIDAD_CAUSAS', 'CAUSAS_DISTINTAS', 'TIPOS_CAUSA_DISTINTOS']

if not ARCHIVO_TABLA_COMPLETA.exists():
    raise FileNotFoundError(f'No se encontró: {ARCHIVO_TABLA_COMPLETA}')

columnas_en_archivo = pd.read_csv(
    ARCHIVO_TABLA_COMPLETA, nrows=0, encoding='utf-8-sig'
).columns.tolist()
if columnas_en_archivo != COLUMNAS_TABLA_COMPLETA:
    raise ValueError(
        'Las columnas de TABLA_COMPLETA.csv no coinciden con el esquema esperado. '
        f'Recibidas: {columnas_en_archivo}'
    )

print(f'Archivo consolidado validado: {ARCHIVO_TABLA_COMPLETA}')


Archivo consolidado validado: data_procesada/tablas_finales/TABLA_COMPLETA.csv


In [4]:
# Configurar y probar exclusivamente la conexión del segundo proyecto
variables_segundo_proyecto = {
    'host2': os.getenv('host2'),
    'port2': os.getenv('port2'),
    'database2': os.getenv('database2'),
    'user2': os.getenv('user2'),
    'password2': os.getenv('password2'),
}
faltantes = [nombre for nombre, valor in variables_segundo_proyecto.items() if not valor]
if faltantes:
    raise ValueError(
        'Faltan credenciales del segundo proyecto en .env: ' + ', '.join(faltantes)
    )

CONEXION_TABLA_COMPLETA = {
    'host': variables_segundo_proyecto['host2'],
    'port': int(variables_segundo_proyecto['port2']),
    'dbname': variables_segundo_proyecto['database2'],
    'user': variables_segundo_proyecto['user2'],
    'password': variables_segundo_proyecto['password2'],
    'sslmode': 'require',
    'connect_timeout': 30,
}

with psycopg.connect(**CONEXION_TABLA_COMPLETA) as conexion:
    solo_lectura = conexion.execute('SHOW default_transaction_read_only').fetchone()[0]
    version = conexion.execute('SELECT version()').fetchone()[0]

if solo_lectura.lower() != 'off':
    raise RuntimeError('El segundo proyecto está en modo de solo lectura.')

print('Conexión al segundo proyecto verificada.')
print(version)


Conexión al segundo proyecto verificada.
PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


In [5]:
# Crear la tabla analítica consolidada en PostgreSQL

DDL_TABLA_COMPLETA = {
    'tabla_completa': '''CREATE TABLE IF NOT EXISTS tabla_completa (
        accidente_id integer PRIMARY KEY,
        fecha_hora timestamp NOT NULL,
        latitud real NOT NULL,
        longitud real NOT NULL,
        clase_accidente text,
        localidad text,
        objetivo_grave boolean NOT NULL,
        clima_id integer,
        fecha_hora_clima timestamp,
        temperatura_2m real,
        humedad_relativa_2m real,
        sensacion_termica real,
        precipitacion real,
        lluvia real,
        nubosidad real,
        presion_superficie real,
        velocidad_viento_10m real,
        direccion_viento_10m real,
        geometria_planta text,
        geometria_terreno text,
        geometria_seccion text,
        sentido_via text,
        n_calzadas real,
        n_carriles real,
        superficie_rodadura text,
        estado_via text,
        condicion_via text,
        iluminacion_artificial text,
        semaforo text,
        cantidad_vehiculos integer NOT NULL,
        clases_vehiculo_distintas integer NOT NULL,
        servicios_vehiculo_distintos integer NOT NULL,
        tiene_motocicleta boolean NOT NULL,
        tiene_bicicleta boolean NOT NULL,
        tiene_automovil boolean NOT NULL,
        tiene_bus boolean NOT NULL,
        tiene_camion boolean NOT NULL,
        cantidad_actores integer NOT NULL,
        edad_promedio real,
        edad_minima real,
        edad_maxima real,
        cantidad_conductores integer NOT NULL,
        cantidad_pasajeros integer NOT NULL,
        cantidad_peatones integer NOT NULL,
        cantidad_heridos integer NOT NULL,
        cantidad_muertos integer NOT NULL,
        cantidad_causas integer NOT NULL,
        causas_distintas integer NOT NULL,
        tipos_causa_distintos integer NOT NULL
    )''',
}

with psycopg.connect(**CONEXION_TABLA_COMPLETA) as conexion:
    for sentencia in DDL_TABLA_COMPLETA.values():
        conexion.execute(sentencia)

print('Tabla consolidada creada o verificada en el segundo proyecto.')


Tabla consolidada creada o verificada en el segundo proyecto.


In [6]:
# Revisar el estado previo de TABLA_COMPLETA en el segundo proyecto
with psycopg.connect(**CONEXION_TABLA_COMPLETA) as conexion:
    filas_existentes_tabla_completa = conexion.execute(
        'SELECT COUNT(*) FROM tabla_completa'
    ).fetchone()[0]

print(f'Filas existentes en tabla_completa: {filas_existentes_tabla_completa:,}')


Filas existentes en tabla_completa: 0


In [9]:
# Cargar lentamente la tabla completa para proteger los recursos de Supabase
FILAS_POR_LOTE = 5000
PAUSA_ENTRE_LOTES_SEGUNDOS = 1.5
REVISAR_TAMANO_CADA_LOTES = 10
LIMITE_PARADA_BYTES = 450 * 1024**2
resultado_carga_tabla_completa = []

def contar_filas_archivo(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(1 for _ in archivo) - 1, 0)

def tamano_total_bd(conexion):
    return conexion.execute(
        'SELECT COALESCE(SUM(pg_database_size(datname)), 0) FROM pg_database'
    ).fetchone()[0]

with psycopg.connect(**CONEXION_TABLA_COMPLETA) as conexion:
    tabla = 'tabla_completa'
    ruta = ARCHIVO_TABLA_COMPLETA
    filas_csv = contar_filas_archivo(ruta)
    existentes = conexion.execute(
        sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
    ).fetchone()[0]

    if existentes == filas_csv:
        print(f'{tabla}: omitida ({existentes:,} filas completas)')
        resultado_carga_tabla_completa.append([tabla, 'omitida', existentes, 0])
    else:

        columnas = [columna.lower() for columna in COLUMNAS_TABLA_COMPLETA]
        temporal = f'_carga_{tabla}'
        conexion.execute(
            sql.SQL('CREATE TEMP TABLE {} (LIKE {} INCLUDING DEFAULTS) ON COMMIT DELETE ROWS').format(
                sql.Identifier(temporal),
                sql.Identifier(tabla),
            )
        )

        consulta_copy = sql.SQL(
            'COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE, ENCODING UTF8)'
        ).format(
            sql.Identifier(temporal),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
        )
        consulta_insertar = sql.SQL(
            'INSERT INTO {} ({}) SELECT {} FROM {} ON CONFLICT DO NOTHING'
        ).format(
            sql.Identifier(tabla),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.SQL(', ').join(map(sql.Identifier, columnas)),
            sql.Identifier(temporal),
        )

        inicio = time.time()
        cargadas = existentes
        # La carga siempre avanza en el orden del CSV; al reanudar se omiten las filas ya confirmadas.
        filas_a_omitir = range(1, existentes + 1) if existentes else None
        for lote, bloque in enumerate(pd.read_csv(
            ruta,
            dtype='string',
            encoding='utf-8-sig',
            chunksize=FILAS_POR_LOTE,
            skiprows=filas_a_omitir,
            low_memory=False,
        ), 1):
            buffer = StringIO()
            bloque.to_csv(buffer, index=False)
            buffer.seek(0)

            with conexion.cursor().copy(consulta_copy) as copia:
                copia.write(buffer.getvalue())
            resultado_insercion = conexion.execute(consulta_insertar)
            cargadas += resultado_insercion.rowcount
            conexion.commit()

            if lote % REVISAR_TAMANO_CADA_LOTES == 0:
                tamano_bytes = tamano_total_bd(conexion)
                if tamano_bytes >= LIMITE_PARADA_BYTES:
                    raise RuntimeError(
                        f'Carga detenida por seguridad: PostgreSQL ocupa '
                        f'{tamano_bytes / 1024**2:.1f} MB.'
                    )

            print(
                f'\r{tabla}: '
                f'{cargadas:,} / {filas_csv:,}',
                end='',
                flush=True,
            )
            time.sleep(PAUSA_ENTRE_LOTES_SEGUNDOS)

        tiempo = round(time.time() - inicio, 1)
        cargadas = conexion.execute(
            sql.SQL('SELECT COUNT(*) FROM {}').format(sql.Identifier(tabla))
        ).fetchone()[0]
        print(f' en {tiempo} s')
        resultado_carga_tabla_completa.append([tabla, 'completa' if cargadas == filas_csv else 'incompleta', cargadas, tiempo])

display(pd.DataFrame(
    resultado_carga_tabla_completa,
    columns=['tabla', 'resultado', 'filas_en_supabase', 'segundos'],
))


tabla_completa: 895,833 / 895,833 en 402.4 s


,tabla,resultado,filas_en_supabase,segundos
0,tabla_completa,completa,895833,402.4


In [10]:
# Validar la carga de la tabla completa
def contar_filas_csv(ruta):
    with ruta.open('rb') as archivo:
        return max(sum(bloque.count(b'\n') for bloque in iter(
            lambda: archivo.read(1024 * 1024), b''
        )) - 1, 0)

with psycopg.connect(**CONEXION_TABLA_COMPLETA) as conexion:
    filas_csv = contar_filas_csv(ARCHIVO_TABLA_COMPLETA)
    filas_bd, ids_unicos, ids_nulos = conexion.execute('''
        SELECT COUNT(*), COUNT(DISTINCT accidente_id), COUNT(*) FILTER (WHERE accidente_id IS NULL)
        FROM tabla_completa
    ''').fetchone()
    tamano_bytes = tamano_total_bd(conexion)

display(pd.DataFrame({
    'metrica': ['Filas CSV', 'Filas Supabase', 'ACCIDENTE_ID únicos', 'ACCIDENTE_ID nulos', 'Tamaño PostgreSQL MB'],
    'valor': [filas_csv, filas_bd, ids_unicos, ids_nulos, round(tamano_bytes / 1024**2, 1)],
}))

if filas_csv != filas_bd or filas_bd != ids_unicos or ids_nulos != 0:
    raise ValueError('La validación final de TABLA_COMPLETA no fue satisfactoria.')


,metrica,valor
0,Filas CSV,895833
1,Filas Supabase,895833
2,ACCIDENTE_ID únicos,895833
3,ACCIDENTE_ID nulos,0
4,Tamaño PostgreSQL MB,252.1
